In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import poisson

data = pd.read_csv("Data/results.csv")

wc_countries = ["Canada", "Mexico", "United States", "Algeria", "Argentina", "Australia", "Austria", "Belgium", "Bosnia and Herzegovina", "Brazil", "Cape Verde", "Colombia", "DR Congo", "Croatia", "Curaçao", "Czech Republic", "Ecuador", "Egypt", "England", "France", "Germany", "Ghana", "Haiti", "Iraq", "Iran", "Ivory Coast", "Japan", "Jordan", "Korea Republic", "Morocco", "Netherlands", "New Zealand", "Norway", "Panama", "Paraguay", "Peru", "Qatar", "Saudi Arabia", "Scotland", "Senegal", "South Africa","Spain","Sweden","Switzerland","Tunisia","Turkey","Uruguay","Uzbekistan"]



In [ ]:
team_to_i= {team: i for i, team in enumerate(wc_countries)}

data["home_team"] = data["home_team"].map(team_to_i)
data["away_team"] = data["away_team"].map(team_to_i)

data['home_adv_multiplier'] = np.where(data['neutral'] == True, 0, 1)

In [ ]:
def dixon_coles_nll(params, df, n):
    # Unpack the current guessed parameters
    attack = params[:n]
    defense = params[n:2*n]
    home_adv = params[-2]
    rho = params[-1]

    # Calculate xG for every match in the dataset based on these parameters
    # Notice we multiply home_adv by the multiplier (0 if neutral, 1 if home stadium)
    home_xg = np.exp(attack[df['home_team_idx']] + 
                     defense[df['away_team_idx']] + 
                     (home_adv * df['home_adv_multiplier']))
    
    away_xg = np.exp(attack[df['away_team_idx']] + 
                     defense[df['home_team_idx']])

    # Get the actual goals scored
    x = df['home_score'].values
    y = df['away_score'].values

    # Calculate standard Poisson Log-Likelihood
    ll_home = poisson.logpmf(x, home_xg)
    ll_away = poisson.logpmf(y, away_xg)

    # Apply the Dixon-Coles Rho Adjustment
    tau = np.ones(len(df))
    
    mask_00 = (x == 0) & (y == 0)
    tau[mask_00] = 1 - (home_xg[mask_00] * away_xg[mask_00] * rho)

    mask_01 = (x == 0) & (y == 1)
    tau[mask_01] = 1 + (home_xg[mask_01] * rho)

    mask_10 = (x == 1) & (y == 0)
    tau[mask_10] = 1 + (away_xg[mask_10] * rho)

    mask_11 = (x == 1) & (y == 1)
    tau[mask_11] = 1 - rho

    # Prevent negative values from crashing the log function
    tau = np.maximum(tau, 1e-10)

    # Return the Negative Log-Likelihood (the penalty score)
    return -np.sum(ll_home + ll_away + np.log(tau))

In [ ]:
# Number of World Cup Teams
n = len(wc_countries)

## 1. Setup Initial Guesses (1.0 for stats, 0.0 for modifiers)
initial_guess = np.concatenate([
    np.ones(n),  # Attack
    np.ones(n),  # Defense
    np.array([0.0]),   # Home Advantage
    np.array([0.0])    # Rho
])

# 2. Setup Bounds to keep math realistic
bounds = [(0.01, 5.0)] * (2 * n) + [(0.0, 2.0)] + [(-0.3, 0.3)]

# 3. Setup Constraint (Sum of attacks must equal number of teams)
def constraint_func(params):
    return sum(params[:n]) - n

constraints = [{'type': 'eq', 'fun': constraint_func}]

# 4. Train the Model!
print("Training model... this may take a moment.")
result = minimize(
    dixon_coles_nll,
    initial_guess,
    args=(data, n),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 200}
)

# 5. Extract the optimized parameters into usable variables
opt_attack = result.x[:n]
opt_defense = result.x[n:2*n]
opt_home_adv = result.x[-2]
opt_rho = result.x[-1]

print(f"Model Trained! Optimal Rho: {opt_rho:.4f}")

In [ ]:
def predict_match(home_team, away_team, is_neutral=False, max_goals=6):
    # 1. Look up the numerical IDs for the teams
    home_idx = team_to_i[home_team]
    away_idx = team_to_i[away_team]

    # 2. Apply Home Advantage (or 0 if neutral)
    h_adv = 0 if is_neutral else opt_home_adv

    # 3. Calculate Expected Goals (xG) using our optimized parameters
    home_xg = np.exp(opt_attack[home_idx] + opt_defense[away_idx] + h_adv)
    away_xg = np.exp(opt_attack[away_idx] + opt_defense[home_idx])

    # 4. Generate the Probability Matrix
    matrix = np.zeros((max_goals, max_goals))
    for x in range(max_goals):
        for y in range(max_goals):
            matrix[x, y] = poisson.pmf(x, home_xg) * poisson.pmf(y, away_xg)

    # 5. Apply the Rho adjustment to the 4 core scorelines
    matrix[0, 0] *= max(0, 1 - (home_xg * away_xg * opt_rho))
    matrix[0, 1] *= max(0, 1 + (home_xg * opt_rho))
    matrix[1, 0] *= max(0, 1 + (away_xg * opt_rho))
    matrix[1, 1] *= max(0, 1 - opt_rho)

    # 6. Calculate Match Odds
    home_win = np.tril(matrix, -1).sum()
    draw = np.trace(matrix)
    away_win = np.triu(matrix, 1).sum()

    print(f"--- {home_team} vs {away_team} ---")
    print(f"Home Win: {home_win:.2%}")
    print(f"Draw: {draw:.2%}")
    print(f"Away Win: {away_win:.2%}")
    
    return matrix

# Example: Predict Canada vs USA (Neutral Venue)
predict_match('Canada', 'United States', is_neutral=True)